In [2]:
import json
import pandas as pd

with open('/home/bruno/UESC/DataSUS/DadosTUB/src/dicionario_sinan_tuberculose.json', 'r', encoding='utf-8') as f:
    dicionario = json.load(f)
df = pd.read_csv('/home/bruno/UESC/DataSUS/DadosTUB/data/processed/tuberculose_ilheus_itabuna_limpo_2014_2024.csv')


Importamos os dados filtrados por completude e o dicionário.



Para continuar nossa análise de qualidade, precisamos tratar os códigos criptografados do SINAN para data de nascimento
com objetivo de traduzir e preencher a tabela com informações de mais simples entendimento.


In [5]:
import numpy as np

# 1. CONVERSÃO DE DATAS
colunas_de_data = [col for col in df.columns if 'Data' in col]
for col in colunas_de_data:
    df[col] = pd.to_datetime(df[col], format='%Y%m%d', errors='coerce')

# 2. ENGENHARIA DA IDADE (DESCODIFICAÇÃO DO SINAN)
def descodificar_idade(codigo):
    if pd.isna(codigo):
        return np.nan
    
    codigo = str(codigo).strip()
    if len(codigo) != 4:
        return np.nan
        
    tipo = codigo[0]
    valor = int(codigo[1:])
    
    if tipo == '4':   # 4 = Idade em Anos
        return valor
    elif tipo == '3': # 3 = Idade em Meses (Menor de 1 ano = 0 anos)
        return 0
    elif tipo == '2': # 2 = Idade em Dias (Menor de 1 ano = 0 anos)
        return 0
    elif tipo == '5': # 5 = Idade acima de 100 anos (valor já é em anos)
        return valor
    else:
        return np.nan
        
if 'Idade_Codigo' in df.columns:
    df['Idade_Anos'] = df['Idade_Codigo'].apply(descodificar_idade)
    df.drop(columns=['Idade_Codigo'], inplace=True)


# 3. MAPEAMENTO DE CATEGORIAS (DE-PARA)
mapa_sexo = {'M': 'Masculino', 'F': 'Feminino', 'I': 'Ignorado'}

mapa_encerramento = {
    '1': 'Cura', '2': 'Abandono', '3': 'Óbito por TB', 
    '4': 'Óbito por outras causas', '5': 'Transferência', 
    '6': 'Mudança de Diagnóstico', '7': 'TB-DR', 
    '8': 'Mudança de Esquema', '9': 'Falência', '10': 'Abandono Primário'
}

mapa_entrada = {
    '1': 'Caso Novo', '2': 'Recidiva', '3': 'Reingresso após abandono', 
    '4': 'Não sabe', '5': 'Transferência', '6': 'Pós-óbito'
}

# Aplicamos os mapas apenas se as colunas sobreviveram ao nosso corte de 45% anterior
if 'Sexo' in df.columns:
    df['Sexo'] = df['Sexo'].map(mapa_sexo).fillna(df['Sexo'])
    
if 'Situacao_Encerramento_Final' in df.columns:
    df['Situacao_Encerramento_Final'] = df['Situacao_Encerramento_Final'].map(mapa_encerramento).fillna(df['Situacao_Encerramento_Final'])
    
if 'Tipo_Entrada' in df.columns:
    df['Tipo_Entrada'] = df['Tipo_Entrada'].map(mapa_entrada).fillna(df['Tipo_Entrada'])

print("\nFormatação e Tipagem concluída!")

# Exibimos a nova estrutura de tipos de dados para auditoria
df[['Data_Notificacao', 'Idade_Anos', 'Sexo', 'Situacao_Encerramento_Final']]


Formatação e Tipagem concluída!


/tmp/ipykernel_12663/2473084085.py:56: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Situacao_Encerramento_Final'] = df['Situacao_Encerramento_Final'].map(mapa_encerramento).fillna(df['Situacao_Encerramento_Final'])
/tmp/ipykernel_12663/2473084085.py:59: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Tipo_Entrada'] = df['Tipo_Entrada'].map(mapa_entrada).fillna(df['Tipo_Entrada'])


,Data_Notificacao,Idade_Anos,Sexo,Situacao_Encerramento_Final
0,2015-12-04,57,Feminino,1.0
1,2015-12-04,70,Feminino,1.0
2,2015-12-23,34,Masculino,1.0
3,2015-12-07,54,Masculino,1.0
4,2015-12-07,30,Masculino,1.0
...,...,...,...,...
3426,2017-04-17,53,Masculino,1.0
3427,2017-04-19,36,Masculino,1.0
3428,2017-04-20,62,Masculino,1.0
3429,2017-04-21,36,Feminino,5.0



Para garantirmos a unicidade dos dados, temos que garantir que não existem dois cadastros de pacientes iguais,
duplicados, isso pode atrapalhar nossos cálculos enquanto análise.


In [11]:
# 1. IDENTIFICAÇÃO DE DUPLICATAS EXATAS
total_linhas_antes = len(df)
duplicatas_exatas = df.duplicated().sum()
print(f"\nTotal de registos (pacientes) antes da limpeza: {total_linhas_antes}")
print(f"Linhas duplicadas encontradas: {duplicatas_exatas}")

# 2. REMOÇÃO AUTOMÁTICA
if duplicatas_exatas > 0:
    # keep='first' (padrão) mantém a primeira ocorrência e apaga as cópias
    df.drop_duplicates(inplace=True)
    
    # Reinicia o índice para garantir que não ficam "buracos" na numeração das linhas
    df.reset_index(drop=True, inplace=True)
    
    print(f"Nova volumetria após limpeza: {len(df)} registos únicos.")
else:
    print("\n A base não possui erros de duplicação exata!")

# Exibe início da tabela
display(df.head(3))


Total de registos (pacientes) antes da limpeza: 3431
Linhas duplicadas encontradas: 0

 A base não possui erros de duplicação exata!


,Tipo_Notificacao,Agravo_Codigo,Data_Notificacao,Ano_Notificacao,UF_Notificacao,Municipio_Notificacao,Regional_Notificacao,Data_Diagnostico,Ano_Nascimento,Sexo,...,Agravo_Tabagismo,Teste_Rapido_Molecular,Teste_Sensibilidade,Uso_Antiretroviral,Baciloscopia_Apos_6_Meses,Transferencia_Confirmada,UF_Transferencia,Municipio_Transferencia,Idade_Anos,Dias_Ate_Encerramento
0,2,A169,2015-12-04,2015,29,291360,1385,2015-12-03,1958.0,Feminino,...,2.0,5.0,7.0,0.0,0.0,0.0,NaN,NaN,57,186.0
1,2,A169,2015-12-04,2015,29,291360,1385,2015-12-04,1945.0,Feminino,...,2.0,5.0,5.0,0.0,0.0,0.0,NaN,NaN,70,214.0
2,2,A169,2015-12-23,2015,29,291480,1386,2015-12-05,1981.0,Masculino,...,1.0,5.0,0.0,0.0,0.0,0.0,NaN,NaN,34,205.0



Também existe a possibilidade de duplicatas que não são 100% iguais, vamos criar uma chave base para análise de possíveis duplicações.


In [8]:
# 1. DEFINIÇÃO DA CHAVE SINTÉTICA PARA GARANTIR UNICIDADE
colunas_chave_sintetica = [
    'Idade_Anos', 
    'Sexo', 
    'Raca_Cor', 
    'Municipio_Residencia', 
    'Data_Diagnostico'
]

# 2. BUSCA DE DUPLICATAS
df_suspeitos = df[df.duplicated(subset=colunas_chave_sintetica, keep=False)].copy()

print("--- ANÁLISE DE UNICIDADA ---")
print(f"Total de casos suspeitos de duplicação: {len(df_suspeitos)}")

# 3. EXIBIÇÃO DE SUSPEITOS
if len(df_suspeitos) > 0:
    print("\nPacientes com a mesma 'Impressão Digital'!")
    df_suspeitos.sort_values(by=colunas_chave_sintetica, inplace=True)
    colunas_auditoria = colunas_chave_sintetica + [
        'Data_Notificacao', 'Tipo_Entrada', 'Situacao_Encerramento_Final'
    ]

    display(df_suspeitos[colunas_auditoria])
else:
    print("\nNenhuma duplicata suspeita encontrada.")

--- ANÁLISE DE UNICIDADA ---
Total de casos suspeitos de duplicação: 74

Pacientes com a mesma 'Impressão Digital'!


,Idade_Anos,Sexo,Raca_Cor,Municipio_Residencia,Data_Diagnostico,Data_Notificacao,Tipo_Entrada,Situacao_Encerramento_Final
1125,19,Feminino,2.0,291480,2014-03-23,2014-04-23,1,2.0
1126,19,Feminino,2.0,291480,2014-03-23,2014-07-25,3,2.0
2118,19,Masculino,4.0,291480,2024-04-17,2024-04-17,1,2.0
2119,19,Masculino,4.0,291480,2024-04-17,2024-08-02,3,2.0
3317,19,Masculino,9.0,291480,2017-02-14,2017-02-16,1,1.0
...,...,...,...,...,...,...,...,...
1935,56,Feminino,2.0,291480,2020-09-17,2021-02-04,3,1.0
2467,57,Masculino,2.0,291480,2021-01-25,2021-02-04,1,2.0
2468,57,Masculino,2.0,291480,2021-01-25,2022-08-19,6,3.0
3158,61,Masculino,4.0,291360,2017-05-16,2017-05-16,1,1.0



Para garantir a validade, vou analisar se existe algum caso em que a data de diagnóstico seja anterior à data de encerramento
de atendimento do paciente.


In [10]:
# EVITA VIAGEM NO TEMPO
# Verifica se a data de encerramento é menor (mais antiga) que o diagnóstico
mascara_erro_cronologico = df['Data_Encerramento'] < df['Data_Diagnostico']
erros_cronologicos = df[mascara_erro_cronologico]

df['Dias_Ate_Encerramento'] = (df['Data_Encerramento'] - df['Data_Diagnostico']).dt.days

# Consideraremos suspeito quem encerrou no mesmo dia (0 dias) sem ser óbito/mudança de diagnóstico,
# ou tratamentos absurdamente longos (ex: > 5 anos / 1825 dias)
mascara_tempo_suspeito = (df['Dias_Ate_Encerramento'] < 0) | (df['Dias_Ate_Encerramento'] > 1825)
erros_tempo = df[mascara_tempo_suspeito & df['Data_Encerramento'].notna()]

print(f"\nInconsistências Cronológicas (Encerramento antes do Diagnóstico): {len(erros_cronologicos)} casos")
print(f"Tempos de Tratamento Suspeitos (> 5 anos ou negativos): {len(erros_tempo)} casos")

if len(erros_cronologicos) > 0 or len(erros_tempo) > 0:
    print("\nVisualizando amostra das inconsistências:")
    colunas_vis = ['Data_Diagnostico', 'Data_Encerramento', 'Dias_Ate_Encerramento', 'Situacao_Encerramento_Final']
    df_erros = pd.concat([erros_cronologicos, erros_tempo]).drop_duplicates()
    display(df_erros[colunas_vis].head(10))
else:
    print("\n Nenhuma quebra de regra cronológica encontrada.")


Inconsistências Cronológicas (Encerramento antes do Diagnóstico): 1 casos
Tempos de Tratamento Suspeitos (> 5 anos ou negativos): 1 casos

Visualizando amostra das inconsistências:


,Data_Diagnostico,Data_Encerramento,Dias_Ate_Encerramento,Situacao_Encerramento_Final
151,2015-08-05,2015-07-21,-15.0,2.0



Número de casos após filtragem de validade.


In [12]:
print(f"Volumetria antes do filtro: {len(df)} pacientes")
df = df[~mascara_erro_cronologico & ~mascara_tempo_suspeito].copy()
df.reset_index(drop=True, inplace=True)

print(f"Volumetria após o filtro: {len(df)} pacientes")
print("\nBase de dados pronta!")

Volumetria antes do filtro: 3431 pacientes
Volumetria após o filtro: 3430 pacientes

Base de dados pronta!


In [17]:
import os

# 1. CRIAR A ESTRUTURA DE PASTAS
os.makedirs('data/processed', exist_ok=True)
os.makedirs('src', exist_ok=True)

# 2. GUARDAR O DATASET FINAL
caminho_dados = 'data/processed/tuberculose_ilheus_itabuna_final.csv'
df.to_csv(caminho_dados, index=False, encoding='utf-8')

✅ Arquivo CSV salvo com sucesso em: data/processed/tuberculose_ilheus_itabuna_final.csv
Você já pode abrir o Notebook 02 para a análise!
